In [ ]:
# Edit only these attached Kaggle Input paths.
from pathlib import Path

BM25_WHEEL_PATH = Path(
    "/kaggle/input/datasets/mduy2911/offline-packages/bm25s-0.3.11-py3-none-any.whl"
)
PUBLIC_LEGALIR_PATH = Path("/kaggle/input/<public-legalir-dataset>/public-official.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/selected-contexts")
RERANKER_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/bge-reranker-v2-m3-kaggle")


In [ ]:
# Frozen system constants. Do not couple these to DEV calibration automatically.
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_DECLARED_REVISION = None  # Optional exact revision supplied with the snapshot.

CHUNK_SIZE = 2_000
CHUNK_OVERLAP = 200
BM25_METHOD = "lucene"
BM25_K1 = 1.5
BM25_B = 0.75
TOP_K_CHUNKS = 2_000
DOCUMENT_AGGREGATION = "sum_top_2"
CANDIDATE_DEPTH = 100
SUPPORTING_CHUNKS_PER_DOCUMENT = 2
FINAL_K = 5
RERANKER_BATCH_SIZE = 128
MAX_SEQUENCE_LENGTH_CAP = 8_192
EXPECTED_PUBLIC_SAMPLES = 1_000
EXPECTED_DOCUMENTS = 8_532
EXPECTED_CHUNKS = 199_816

OUTPUT_PATH = Path("/kaggle/working/submission.json")
METADATA_PATH = Path("/kaggle/working/legalir_public_inference_metadata.json")


In [ ]:
# Fail before imports/model loading if an offline artifact is invalid.
import os
import subprocess
import sys

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if not BM25_WHEEL_PATH.is_file():
    raise FileNotFoundError(f"Attach the local BM25 wheel at: {BM25_WHEEL_PATH}")
if not PUBLIC_LEGALIR_PATH.is_file():
    raise FileNotFoundError(f"Attach the official public LegalIR JSON at: {PUBLIC_LEGALIR_PATH}")
if not CORPUS_PATH.is_dir():
    raise FileNotFoundError(f"Attach the LegalIR corpus directory at: {CORPUS_PATH}")
if not RERANKER_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        f"Attach the complete local reranker snapshot at: {RERANKER_MODEL_PATH}"
    )
if OUTPUT_PATH == METADATA_PATH:
    raise ValueError("prediction path and metadata path must differ")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        str(BM25_WHEEL_PATH),
    ],
    check=True,
)


In [ ]:
import json
import re
from collections import Counter, defaultdict
from math import isfinite
from time import perf_counter

import bm25s
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def reject_duplicate_object_keys(pairs):
    value = {}
    for key, item in pairs:
        if key in value:
            raise ValueError(f"duplicate JSON object key: {key!r}")
        value[key] = item
    return value


def read_json_strict(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream, object_pairs_hook=reject_duplicate_object_keys)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_public_legalir(path: Path) -> dict:
    value = read_json_strict(path)
    if not isinstance(value, dict):
        raise ValueError(f"{path}: expected a top-level object keyed by sample ID")
    if len(value) != EXPECTED_PUBLIC_SAMPLES:
        raise ValueError(
            f"{path}: expected exactly {EXPECTED_PUBLIC_SAMPLES} samples, got {len(value)}"
        )
    canonical = {}
    for sample_id, sample in value.items():
        canonical_id = str(sample_id)
        if canonical_id in canonical:
            raise ValueError(f"duplicate sample ID after canonicalization: {canonical_id}")
        if not isinstance(sample, dict):
            raise TypeError(f"sample {canonical_id!r}: expected an object")
        if not isinstance(sample.get("question"), str):
            raise TypeError(f"sample {canonical_id!r}: question must be a string")
        if "answer" not in sample or sample["answer"] is not None:
            raise ValueError(
                f"sample {canonical_id!r}: public answer must be explicit null"
            )
        canonical[canonical_id] = sample
    return canonical


def load_corpus(path: Path) -> list[dict]:
    json_paths = sorted(
        item for item in path.rglob("*") if item.is_file() and item.suffix.lower() == ".json"
    )
    if not json_paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in json_paths:
        value = read_json_strict(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)
    document_ids = [str(document.get("id")) for document in documents]
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    return documents


def chunk_corpus(documents: list[dict]) -> list[dict]:
    if CHUNK_SIZE <= 0 or CHUNK_OVERLAP < 0 or CHUNK_OVERLAP >= CHUNK_SIZE:
        raise ValueError("invalid fixed-window chunk parameters")
    step = CHUNK_SIZE - CHUNK_OVERLAP
    chunks = []
    for document in documents:
        document_id = str(document["id"])
        passage = document.get("passage")
        if not isinstance(passage, str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        if not passage:
            continue
        for chunk_index, start in enumerate(range(0, len(passage), step)):
            end = min(start + CHUNK_SIZE, len(passage))
            chunks.append(
                {
                    "chunk_id": f"{document_id}:{chunk_index}",
                    "document_id": document_id,
                    "text": passage[start:end],
                }
            )
            if end == len(passage):
                break
    return chunks


TOKEN_PATTERN = re.compile(r"\w+", flags=re.UNICODE)


def lexical_tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


def aggregate_sum_top_2(chunk_hits: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    best_chunk_rank = {}
    for hit in chunk_hits:
        document_id = hit["document_id"]
        score = float(hit["score"])
        if not isfinite(score):
            raise ValueError("BM25 score must be finite")
        grouped[document_id].append(score)
        best_chunk_rank[document_id] = min(
            best_chunk_rank.get(document_id, hit["rank"]), hit["rank"]
        )
    ranked = [
        {
            "document_id": document_id,
            "score": sum(sorted(scores, reverse=True)[:2]),
            "best_chunk_rank": best_chunk_rank[document_id],
        }
        for document_id, scores in grouped.items()
    ]
    ranked.sort(
        key=lambda item: (-item["score"], item["best_chunk_rank"], item["document_id"])
    )
    return ranked


def build_bm25(chunks: list[dict]):
    if bm25s.__version__ != "0.3.11":
        raise RuntimeError(f"expected bm25s==0.3.11, got {bm25s.__version__}")
    started = perf_counter()
    tokenized = bm25s.tokenize(
        [chunk["text"] for chunk in chunks],
        lower=True,
        token_pattern=r"(?u)\w+",
        stopwords=[],
        stemmer=None,
        return_ids=True,
        show_progress=False,
    )
    retriever = bm25s.BM25(k1=BM25_K1, b=BM25_B, method=BM25_METHOD)
    retriever.index(tokenized, show_progress=False)
    return {"retriever": retriever, "seconds": perf_counter() - started}


def retrieve_fixed_candidates(retriever, chunks: list[dict], samples: dict) -> dict:
    sample_items = list(samples.items())
    candidates_by_query = {}
    started = perf_counter()
    for batch_start in range(0, len(sample_items), 64):
        batch = sample_items[batch_start : batch_start + 64]
        result = retriever.retrieve(
            [lexical_tokenize(sample["question"]) for _, sample in batch],
            k=TOP_K_CHUNKS,
            sorted=True,
            return_as="tuple",
            show_progress=False,
        )
        for (sample_id, _), hit_indices, hit_scores in zip(
            batch, result.documents, result.scores
        ):
            chunk_hits = []
            for rank, (index_value, score_value) in enumerate(
                zip(hit_indices, hit_scores), start=1
            ):
                index = int(index_value)
                chunk = chunks[index]
                chunk_hits.append(
                    {
                        "chunk_index": index,
                        "document_id": chunk["document_id"],
                        "score": float(score_value),
                        "rank": rank,
                    }
                )
            selected = aggregate_sum_top_2(chunk_hits)[:CANDIDATE_DEPTH]
            if len(selected) != CANDIDATE_DEPTH:
                raise RuntimeError(
                    f"sample {sample_id!r}: expected {CANDIDATE_DEPTH} candidates"
                )
            support = {item["document_id"]: [] for item in selected}
            for hit in chunk_hits:
                values = support.get(hit["document_id"])
                if values is not None and len(values) < SUPPORTING_CHUNKS_PER_DOCUMENT:
                    chunk = chunks[hit["chunk_index"]]
                    values.append(
                        {
                            "chunk_id": chunk["chunk_id"],
                            "text": chunk["text"],
                            "bm25_rank": hit["rank"],
                            "bm25_score": hit["score"],
                        }
                    )
            candidates = []
            for original_rank, item in enumerate(selected, start=1):
                supporting_chunks = support[item["document_id"]]
                if not 1 <= len(supporting_chunks) <= 2:
                    raise RuntimeError("candidate must have one or two supporting chunks")
                candidates.append(
                    {
                        "document_id": item["document_id"],
                        "original_rank": original_rank,
                        "supporting_chunks": supporting_chunks,
                    }
                )
            candidate_ids = [item["document_id"] for item in candidates]
            if len(candidate_ids) != len(set(candidate_ids)):
                raise RuntimeError(f"sample {sample_id!r}: duplicate candidate IDs")
            candidates_by_query[sample_id] = candidates
    return {"candidates": candidates_by_query, "seconds": perf_counter() - started}


def local_model_metadata(model):
    config_commit_hash = getattr(model.config, "_commit_hash", None)
    verified = isinstance(config_commit_hash, str) and bool(config_commit_hash.strip())
    return {
        "model_name": RERANKER_MODEL_NAME,
        "local_input_path": str(RERANKER_MODEL_PATH),
        "declared_revision": RERANKER_DECLARED_REVISION,
        "config_commit_hash": config_commit_hash if verified else None,
        "revision_status": (
            "verified-from-config" if verified else "declared-offline-snapshot"
        ),
    }


def load_reranker() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("This public inference notebook requires a Kaggle CUDA accelerator")
    torch.cuda.reset_peak_memory_stats()
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        RERANKER_MODEL_PATH,
        local_files_only=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_PATH,
        dtype=torch.float16,
        local_files_only=True,
    )
    model.to("cuda")
    model.eval()
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(
        getattr(model.config, "max_position_embeddings", tokenizer_limit)
    )
    max_length = min(tokenizer_limit, model_limit, MAX_SEQUENCE_LENGTH_CAP)
    if max_length <= 0:
        raise ValueError("resolved reranker maximum length must be positive")
    return {
        "tokenizer": tokenizer,
        "model": model,
        "max_length": max_length,
        "load_seconds": perf_counter() - started,
        "metadata": local_model_metadata(model),
    }


def score_pairs(reranker: dict, pairs: list[tuple[str, str]]) -> dict:
    scores = []
    started = perf_counter()
    for batch_start in range(0, len(pairs), RERANKER_BATCH_SIZE):
        batch = pairs[batch_start : batch_start + RERANKER_BATCH_SIZE]
        encoded = reranker["tokenizer"](
            [question for question, _ in batch],
            [passage for _, passage in batch],
            padding=True,
            truncation="only_second",
            max_length=reranker["max_length"],
            return_tensors="pt",
        )
        encoded = {name: value.to("cuda") for name, value in encoded.items()}
        with torch.no_grad():
            logits = reranker["model"](
                **encoded,
                return_dict=True,
            ).logits.view(-1).float()
        batch_scores = logits.cpu().tolist()
        if len(batch_scores) != len(batch) or not all(
            isfinite(value) for value in batch_scores
        ):
            raise RuntimeError("reranker returned invalid scores")
        scores.extend(float(value) for value in batch_scores)
    return {"scores": scores, "seconds": perf_counter() - started}


def rerank_candidates(reranker: dict, samples: dict, candidates_by_query: dict) -> dict:
    pairs = []
    layout = []
    for sample_id, sample in samples.items():
        candidates = candidates_by_query[sample_id]
        counts = []
        for candidate in candidates:
            counts.append(len(candidate["supporting_chunks"]))
            pairs.extend(
                (sample["question"], supporting["text"])
                for supporting in candidate["supporting_chunks"]
            )
        layout.append((sample_id, counts))

    scoring = score_pairs(reranker, pairs)
    offset = 0
    rankings = {}
    for sample_id, counts in layout:
        candidates = candidates_by_query[sample_id]
        reranked = []
        for candidate, count in zip(candidates, counts):
            chunk_scores = scoring["scores"][offset : offset + count]
            offset += count
            reranked.append(
                {
                    "document_id": candidate["document_id"],
                    "score": sum(chunk_scores),
                    "original_rank": candidate["original_rank"],
                }
            )
        reranked.sort(
            key=lambda item: (-item["score"], item["original_rank"], item["document_id"])
        )
        ranking = [item["document_id"] for item in reranked]
        candidate_ids = [item["document_id"] for item in candidates]
        if len(ranking) != len(set(ranking)):
            raise RuntimeError(f"sample {sample_id!r}: duplicate ranked IDs")
        if set(ranking) != set(candidate_ids):
            raise RuntimeError("reranking changed the candidate universe")
        rankings[sample_id] = ranking
    if offset != len(scoring["scores"]):
        raise RuntimeError("not every cross-encoder score was consumed")
    return {
        "rankings": rankings,
        "number_of_pairs": len(pairs),
        "seconds": scoring["seconds"],
    }


def build_predictions(public_samples: dict, rankings: dict) -> dict:
    predictions = {}
    for sample_id in public_samples:
        ranking = rankings.get(sample_id)
        if ranking is None:
            raise KeyError(f"sample {sample_id!r}: missing ranking")
        if len(ranking) != len(set(ranking)):
            raise ValueError(f"sample {sample_id!r}: duplicate ranked IDs")
        answer = ranking[:FINAL_K]
        if len(answer) != FINAL_K:
            raise ValueError(
                f"sample {sample_id!r}: expected exactly {FINAL_K} ranked documents"
            )
        if len(answer) != len(set(answer)):
            raise ValueError(f"sample {sample_id!r}: duplicate answer IDs")
        predictions[sample_id] = {"answer": answer}
    return predictions


def validate_predictions(predictions: dict, public_samples: dict, corpus_ids: set[str]):
    prediction_ids = set(predictions)
    input_ids = set(public_samples)
    missing_ids = input_ids - prediction_ids
    extra_ids = prediction_ids - input_ids
    duplicate_answer_count = 0
    answer_lengths = Counter()

    for sample_id, value in predictions.items():
        if not isinstance(value, dict) or set(value) != {"answer"}:
            raise ValueError(f"sample {sample_id!r}: output must contain answer only")
        answer = value["answer"]
        if not isinstance(answer, list):
            raise TypeError(f"sample {sample_id!r}: answer must be a list")
        if not 1 <= len(answer) <= 5:
            raise ValueError(f"sample {sample_id!r}: answer length must be 1..5")
        if len(answer) != len(set(answer)):
            duplicate_answer_count += 1
        if not all(isinstance(document_id, str) for document_id in answer):
            raise TypeError(f"sample {sample_id!r}: document IDs must be strings")
        unknown = set(answer) - corpus_ids
        if unknown:
            raise ValueError(
                f"sample {sample_id!r}: returned document IDs absent from corpus"
            )
        answer_lengths[len(answer)] += 1

    if missing_ids or extra_ids:
        raise ValueError("prediction IDs do not exactly match public input IDs")
    if duplicate_answer_count:
        raise ValueError("duplicate document IDs found within prediction answers")
    return {
        "answer_length_distribution": {
            str(length): answer_lengths[length] for length in sorted(answer_lengths)
        },
        "duplicate_answer_count": duplicate_answer_count,
        "missing_id_count": len(missing_ids),
        "extra_id_count": len(extra_ids),
    }


In [ ]:
# Offline preflight and synthetic reranker smoke check.
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
assert TOP_K_CHUNKS == 2_000
assert DOCUMENT_AGGREGATION == "sum_top_2"
assert CANDIDATE_DEPTH == 100
assert SUPPORTING_CHUNKS_PER_DOCUMENT == 2
assert FINAL_K == 5

reranker = load_reranker()
smoke = score_pairs(
    reranker,
    [("Câu hỏi kiểm tra.", "Đoạn văn kiểm tra.")],
)
assert len(smoke["scores"]) == 1
assert isfinite(smoke["scores"][0])
print("Offline preflight and reranker smoke check passed.")


In [ ]:
run_started = perf_counter()
public_samples = load_public_legalir(PUBLIC_LEGALIR_PATH)
documents = load_corpus(CORPUS_PATH)
chunks = chunk_corpus(documents)
if len(documents) != EXPECTED_DOCUMENTS or len(chunks) != EXPECTED_CHUNKS:
    raise ValueError(
        f"expected {EXPECTED_DOCUMENTS:,} documents / {EXPECTED_CHUNKS:,} chunks, "
        f"got {len(documents):,} / {len(chunks):,}"
    )

bm25 = build_bm25(chunks)
fixed_candidates = retrieve_fixed_candidates(
    bm25["retriever"],
    chunks,
    public_samples,
)
reranked = rerank_candidates(
    reranker,
    public_samples,
    fixed_candidates["candidates"],
)
predictions = build_predictions(public_samples, reranked["rankings"])
corpus_ids = {str(document["id"]) for document in documents}
validation = validate_predictions(predictions, public_samples, corpus_ids)

assert set(predictions) == set(public_samples)
assert all(isinstance(value["answer"], list) for value in predictions.values())
assert all(1 <= len(value["answer"]) <= 5 for value in predictions.values())
assert all(
    len(value["answer"]) == len(set(value["answer"]))
    for value in predictions.values()
)
assert all(
    document_id in corpus_ids
    for value in predictions.values()
    for document_id in value["answer"]
)

runtime = {
    "model_load_seconds": reranker["load_seconds"],
    "bm25_build_seconds": bm25["seconds"],
    "candidate_retrieval_seconds": fixed_candidates["seconds"],
    "reranking_seconds": reranked["seconds"],
    "cross_encoder_pairs": reranked["number_of_pairs"],
    "total_seconds": perf_counter() - run_started,
    "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}

metadata = {
    "input_sample_count": len(public_samples),
    "output_sample_count": len(predictions),
    "chunk_configuration": {
        "chunk_size": CHUNK_SIZE,
        "overlap": CHUNK_OVERLAP,
        "documents": len(documents),
        "chunks": len(chunks),
    },
    "bm25_configuration": {
        "library": f"bm25s=={bm25s.__version__}",
        "method": BM25_METHOD,
        "k1": BM25_K1,
        "b": BM25_B,
        "tokenization": r"lowercase Unicode \w+",
        "top_k_chunks": TOP_K_CHUNKS,
    },
    "aggregation": {
        "bm25_document_score": "sum top-2 BM25 chunk scores",
        "reranker_document_score": "sum up to 2 independent cross-encoder chunk scores",
    },
    "candidate_depth": CANDIDATE_DEPTH,
    "reranker_model": RERANKER_MODEL_NAME,
    "declared_offline_model_metadata": reranker["metadata"],
    "batch_size": RERANKER_BATCH_SIZE,
    "runtime": runtime,
    "prediction_answer_length_distribution": validation[
        "answer_length_distribution"
    ],
    "duplicate_answer_count": validation["duplicate_answer_count"],
    "missing_id_count": validation["missing_id_count"],
    "extra_id_count": validation["extra_id_count"],
}

OUTPUT_PATH.write_text(
    json.dumps(predictions, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(metadata, ensure_ascii=False, indent=2))
print("Saved predictions:", OUTPUT_PATH)
print("Saved metadata:", METADATA_PATH)
print("Only submission.json is a submission payload.")
print("Do not include the metadata JSON in submission.zip.")
